In [15]:

GITHUB_REPO = "https://github.com/kshitizrajpatel/collaborative_cnn_team05.git"
MODEL_SUBDIR = "models"
MODEL_DEF_FILENAME = "model_v1.py"
MODEL_WEIGHTS_FILENAME = "model_v1.pth"
MODEL_BRANCH = "main"
IMG_SIZE_DEFAULT = 224
BATCH_SIZE = 32



In [16]:
!pip install -q kaggle torchvision
print("Installed kaggle and torchvision.")


Installed kaggle and torchvision.


In [17]:
import os, shutil, glob

if os.path.exists("repo"):
    shutil.rmtree("repo")


print("Cloning repository (shallow)...")
!git clone --depth 1 {GITHUB_REPO} repo || true


os.makedirs("repo/models", exist_ok=True)

raw_base = GITHUB_REPO.replace("github.com", "raw.githubusercontent.com").rstrip(".git")
model_py_url = f"{raw_base}/{MODEL_BRANCH}/{MODEL_SUBDIR}/{MODEL_DEF_FILENAME}"
model_pth_url = f"{raw_base}/{MODEL_BRANCH}/{MODEL_SUBDIR}/{MODEL_WEIGHTS_FILENAME}"

print("Downloading model definition from:", model_py_url)
!wget -q -O repo/models/{MODEL_DEF_FILENAME} "{model_py_url}" || true

print("Downloading model weights from:", model_pth_url)
!wget -q -O repo/models/{MODEL_WEIGHTS_FILENAME} "{model_pth_url}" || true

print("Files in repo/models:", glob.glob("repo/models/*"))


Cloning repository (shallow)...
Cloning into 'repo'...
remote: Enumerating objects: 37, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 37 (delta 5), reused 35 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (37/37), 16.45 MiB | 13.92 MiB/s, done.
Resolving deltas: 100% (5/5), done.
Files in repo/models: ['repo/models/model_v1.py', 'repo/models/model_v1.pth']


In [18]:
from google.colab import files
import os, glob

def ensure_file(path, prompt_name):
    if not os.path.exists(path):
        print(f"{path} not found. Please upload {prompt_name} now.")
        uploaded = files.upload()
        for name in uploaded:
            os.rename(name, path)
        if not os.path.exists(path):
            raise FileNotFoundError(f"{path} is still missing after upload.")

MODEL_DEF_PATH = os.path.join("repo", "models", MODEL_DEF_FILENAME)
MODEL_WEIGHTS_PATH = os.path.join("repo", "models", MODEL_WEIGHTS_FILENAME)

ensure_file(MODEL_DEF_PATH, MODEL_DEF_FILENAME)
ensure_file(MODEL_WEIGHTS_PATH, MODEL_WEIGHTS_FILENAME)

print("Model files present:")
print(MODEL_DEF_PATH)
print(MODEL_WEIGHTS_PATH)


Model files present:
repo/models/model_v1.py
repo/models/model_v1.pth


In [21]:
import torch, importlib.util, inspect, os, traceback
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
MODEL_DEF_PATH = "repo/models/model_v1.py"
MODEL_WEIGHTS_PATH = "repo/models/model_v1.pth"

spec = importlib.util.spec_from_file_location("user_model_module", MODEL_DEF_PATH)
mod = importlib.util.module_from_spec(spec)
try:
    spec.loader.exec_module(mod)
except Exception as e:
    traceback.print_exc()
    raise RuntimeError(f"Failed to import {MODEL_DEF_PATH}: {e}")

if not hasattr(mod, "create_mobilenet_model"):
    raise RuntimeError("create_mobilenet_model() not found in model_v1.py — make sure the function name matches exactly.")

create_fn = getattr(mod, "create_mobilenet_model")
if not inspect.isfunction(create_fn):
    raise RuntimeError("create_mobilenet_model exists but is not a function.")

print("Imported create_mobilenet_model from model_v1.py")

ckpt = torch.load(MODEL_WEIGHTS_PATH, map_location="cpu")
if isinstance(ckpt, dict):
    if "state_dict" in ckpt:
        sd = ckpt["state_dict"]
    elif "model_state_dict" in ckpt:
        sd = ckpt["model_state_dict"]
    elif any(torch.is_tensor(v) for v in ckpt.values()):
        sd = ckpt
    else:
        torch.save(ckpt, "extracted_checkpoint_debug.pth")
        raise RuntimeError("Checkpoint dict format not recognized. Saved to extracted_checkpoint_debug.pth for inspection.")
else:
    if isinstance(ckpt, torch.nn.Module):
        model = ckpt.to(device)
        model.eval()
        print("Checkpoint was a full model object; model loaded and ready.")
        sd = None
    else:
        raise RuntimeError(f"Unsupported checkpoint type: {type(ckpt)}")

def clean_state_dict(sd):
    new = {}
    for k, v in sd.items():
        if k.startswith("module."):
            new[k.replace("module.", "", 1)] = v
        else:
            new[k] = v
    return new

if 'sd' in locals() and sd is not None:
    sd_clean = clean_state_dict(sd)
    inferred_num_classes = None
    candidates = [
        "classifier.1.weight",
        "classifier.weight",
        "classifier.fc.weight",
        "fc.weight",
        "head.weight"
    ]
    for key in candidates:
        if key in sd_clean:
            weight = sd_clean[key]
            try:
                inferred_num_classes = weight.shape[0]
                print(f"Inferred num_classes={inferred_num_classes} from checkpoint key '{key}' (shape={weight.shape})")
                break
            except Exception:
                continue

    if inferred_num_classes is None:
        inferred_num_classes = 38
        print(f"Could not infer num_classes from checkpoint; falling back to default num_classes={inferred_num_classes}")

    try:
        model = create_fn(inferred_num_classes)
        print(f"Instantiated MobileNet model with num_classes={inferred_num_classes}")
    except Exception as e:
        traceback.print_exc()
        raise RuntimeError(f"create_mobilenet_model({inferred_num_classes}) failed: {e}")

    try:
        model.load_state_dict(sd_clean, strict=False)
        model.to(device)
        model.eval()
        print("Loaded state_dict into model (strict=False). Model ready.")
    except Exception as e:
        torch.save(sd_clean, "extracted_state_dict.pth")
        traceback.print_exc()
        raise RuntimeError("Failed to load state_dict into model. Saved cleaned state dict to extracted_state_dict.pth for inspection.") from e


Device: cpu
Imported create_mobilenet_model from model_v1.py
Inferred num_classes=2 from checkpoint key 'classifier.1.weight' (shape=torch.Size([2, 1280]))
Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 114MB/s]

Instantiated MobileNet model with num_classes=2
Loaded state_dict into model (strict=False). Model ready.


In [22]:
from google.colab import files
import os

print("Please upload kaggle.json (Kaggle API token).")
uploaded = files.upload()
if 'kaggle.json' in uploaded:
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    with open(os.path.expanduser("~/.kaggle/kaggle.json"), "wb") as f:
        f.write(uploaded['kaggle.json'])
    os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
else:
    print("kaggle.json not uploaded — ensure it is available at /root/.kaggle/kaggle.json if you skipped upload.")

DATA_DIR = "data_new_plant"
os.makedirs(DATA_DIR, exist_ok=True)
print("Downloading Kaggle dataset (this may take several minutes)...")
!kaggle datasets download -d vipoooool/new-plant-diseases-dataset -p "{DATA_DIR}" --unzip
print("Download complete.")


Please upload kaggle.json (Kaggle API token).


Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/vipoooool/new-plant-diseases-dataset
License(s): copyright-authors
100% 2.69G/2.70G [00:32<00:00, 165MB/s] 
100% 2.70G/2.70G [00:32<00:00, 88.9MB/s]
Download complete.


In [23]:
import os, pandas as pd, glob
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Dataset
from PIL import Image

possible_test_paths = []
for root, dirs, files in os.walk(DATA_DIR):
    for d in dirs:
        if d.lower().startswith("test"):
            possible_test_paths.append(os.path.join(root, d))
possible_test_paths = list(dict.fromkeys(possible_test_paths))
print("Possible test folders:", possible_test_paths)

test_dir = None
for p in possible_test_paths:
    if any(f.lower().endswith((".jpg",".jpeg",".png")) for f in os.listdir(p)):
        test_dir = p
        break
if test_dir is None and possible_test_paths:
    test_dir = possible_test_paths[0]
if test_dir is None:
    raise FileNotFoundError("Could not find a test folder in the downloaded dataset.")

print("Using test folder:", test_dir)

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

class FlatTestDataset(Dataset):
    def __init__(self, folder, transform=None, labels_map=None):
        self.paths = sorted([os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith((".jpg",".jpeg",".png"))])
        self.transform = transform
        self.fnames = [os.path.basename(p) for p in self.paths]
        self.labels_map = labels_map or {}

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        p = self.paths[idx]
        img = Image.open(p).convert("RGB")
        img_t = self.transform(img) if self.transform else transforms.ToTensor()(img)
        meta = {"filename": self.fnames[idx], "path": p}
        if self.fnames[idx] in self.labels_map:
            meta["ground_truth"] = self.labels_map[self.fnames[idx]]
        return img_t, meta

using_custom = False
try:
    imgfolder = ImageFolder(test_dir, transform=transform)
    if len(imgfolder) > 0 and len(imgfolder.classes) > 0:
        print("Detected class subfolders. Using ImageFolder with classes:", imgfolder.classes)
        test_loader = DataLoader(imgfolder, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
        class_names = imgfolder.classes
        using_custom = False
    else:
        raise Exception("ImageFolder empty or no classes")
except Exception as e:
    print("ImageFolder not used (flat test). Reason:", e)
    labels_csv = None
    for fname in ("labels.csv","test_labels.csv","GT.csv","test.csv"):
        cand = os.path.join(os.path.dirname(test_dir), fname)
        if os.path.exists(cand):
            labels_csv = cand
            break
    if labels_csv is None:
        for fname in ("labels.csv","test_labels.csv","GT.csv","test.csv"):
            cand = os.path.join(test_dir, fname)
            if os.path.exists(cand):
                labels_csv = cand
                break
    labels_map = {}
    if labels_csv:
        try:
            df = pd.read_csv(labels_csv)
            possible_fname_cols = [c for c in df.columns if c.lower() in ("image","filename","file","img")]
            fname_col = possible_fname_cols[0] if possible_fname_cols else df.columns[0]
            possible_label_cols = [c for c in df.columns if c.lower() in ("label","class","diagnosis","species")]
            label_col = possible_label_cols[0] if possible_label_cols else (df.columns[1] if df.shape[1] > 1 else df.columns[0])
            for _, row in df.iterrows():
                labels_map[str(row[fname_col])] = str(row[label_col])
            print("Loaded labels from CSV:", labels_csv)
        except Exception as e:
            print("Could not parse labels CSV:", e)
    custom_dataset = FlatTestDataset(test_dir, transform=transform, labels_map=labels_map)
    test_loader = DataLoader(custom_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    using_custom = True
    class_names = None

print("Dataloader prepared. using_custom:", using_custom)


Possible test folders: ['data_new_plant/test', 'data_new_plant/test/test']
Using test folder: data_new_plant/test/test
ImageFolder not used (flat test). Reason: Couldn't find any class folder in data_new_plant/test/test.
Dataloader prepared. using_custom: True


In [25]:
import torch, json
import torch.nn.functional as F
from google.colab import files as colab_files
import os

model.eval()
softmax = torch.nn.Softmax(dim=1)
results = []

def build_result_entry(fname, probs, class_names=None, gt=None, topk_k=5):
    # probs: list of floats for each class
    import torch
    probs_tensor = torch.tensor(probs)
    k = min(topk_k, probs_tensor.size(0))
    topk = torch.topk(probs_tensor, k=k).indices.tolist()
    top5 = [{"index": int(idx), "prob": float(probs[idx])} for idx in topk]
    pred_idx = int(topk[0]) if len(topk) > 0 else None
    pred_name = class_names[pred_idx] if (class_names and pred_idx is not None and pred_idx < len(class_names)) else None
    return {
        "filename": fname,
        "predicted_index": pred_idx,
        "predicted_label": pred_name,
        "top5": top5,
        "ground_truth": gt
    }

with torch.no_grad():
    if using_custom:
        for batch in test_loader:
            if isinstance(batch, (list, tuple)) and len(batch) == 2:
                inputs, metas = batch
            else:
                try:
                    inputs = batch[0]
                    metas = batch[1]
                except Exception:
                    raise RuntimeError("Unexpected batch structure from custom test_loader; please print a sample batch to inspect.")

            inputs = inputs.to(device)
            outputs = model(inputs)
            probs = softmax(outputs).cpu().tolist()

            for i in range(len(probs)):
                p = probs[i]
                meta_item = None
                fname = None
                gt = None

                if isinstance(metas, (list, tuple)):
                    if i < len(metas):
                        meta_item = metas[i]
                    else:
                        meta_item = None
                else:
                    meta_item = metas

                if isinstance(meta_item, dict):
                    fname = meta_item.get("filename") or meta_item.get("file") or meta_item.get("image")
                    gt = meta_item.get("ground_truth", None)
                elif isinstance(meta_item, str):
                    fname = meta_item
                elif isinstance(meta_item, (list, tuple)) and len(meta_item) > 0 and isinstance(meta_item[0], str):
                    # sometimes collate returns a tuple of filenames
                    fname = meta_item[0]
                else:
                    # fallback: use dataset paths if available
                    try:
                        # attempt to access dataset attribute (works if test_loader.dataset has paths)
                        ds = test_loader.dataset
                        if hasattr(ds, "fnames") and i < len(ds.fnames):
                            fname = ds.fnames[i]
                        elif hasattr(ds, "paths") and i < len(ds.paths):
                            fname = os.path.basename(ds.paths[i])
                        else:
                            fname = f"img_{len(results)}"
                    except Exception:
                        fname = f"img_{len(results)}"

                entry = build_result_entry(fname, p, class_names if 'class_names' in globals() else None, gt)
                results.append(entry)

    else:
        if not hasattr(test_loader.dataset, "_iter_ptr"):
            test_loader.dataset._iter_ptr = 0
        for inputs, _ in test_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            probs = softmax(outputs).cpu().tolist()
            ptr = test_loader.dataset._iter_ptr
            for i in range(inputs.size(0)):
                sample_path, _ = test_loader.dataset.samples[ptr + i]
                fname = os.path.basename(sample_path)
                p = probs[i]
                entry = build_result_entry(fname, p, test_loader.dataset.classes if hasattr(test_loader.dataset, "classes") else None, gt=None)
                results.append(entry)
            test_loader.dataset._iter_ptr += inputs.size(0)

OUT_JSON = "results.json"
with open(OUT_JSON, "w") as f:
    json.dump(results, f, indent=2)

print(f"Saved {len(results)} predictions to {OUT_JSON}")
colab_files.download(OUT_JSON)


Saved 33 predictions to results.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>